<div dir="rtl" style="text-align:right">

# 👁 بینایی ماشین

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SharifiZarchi/IntroAI/blob/main/Session_08/ComputerVision/Computer_Vision_Tutorial_FA.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FSharifiZarchi%2FIntroAI%2Fblob%2Fmain%2FSession_08%2FComputerVision%2FComputer_Vision_Tutorial_FA.ipynb)

این جلسه به عملیات و معماری‌ای می‌پردازد که پشت بینایی ماشین مدرن است: **کانولوشن** و **شبکه عصبی پیچشی (CNN)**، از پیاده‌سازی numpy تا مدل‌های ازپیش‌آموزش‌دیده سطح صنعتی برای دسته‌بندی، آشکارسازی و قطعه‌بندی. مسیر:

۱. عملیات کانولوشن، پیاده‌سازی از صفر

۲. از کرنل تا لایه: بلوک‌های سازنده CNN

۳. یک CNN روی CIFAR-10، ارزیابی‌شده در برابر شبکه تمام‌متصل جلسه ۷

۴. درون CNN آموزش‌دیده: کرنل‌ها و نقشه‌های ویژگی

۵. داده‌افزایی با albumentations

۶. یادگیری انتقالی با ResNet18

۷. ابزارهای حرفه‌ای: آشکارسازی شیء و قطعه‌بندی معنایی

۸. جمع‌بندی و گام‌های بعدی

پیش‌نیازها: جلسه ۷ (حلقه آموزش PyTorch و نتیجه شبکه تمام‌متصل روی CIFAR-10). **در Colab** همه‌چیز از قبل نصب است؛ GPU (مسیر *Runtime ▸ Change runtime type ▸ T4 GPU*) بخش‌های ۳ و ۶ را چند برابر سریع‌تر می‌کند، هرچند CPU در سراسر نوت‌بوک کار می‌کند (سلول‌های کند هر کدام حدود ۲ دقیقه‌اند و سلول ریزتنظیم روی CPU حدود ده دقیقه در برابر حدود یک دقیقه روی GPU).

**روش اجرا:** روی هر سلول کلیک کن و `Shift + Enter` بزن، به ترتیب از بالا به پایین.

</div>

<div dir="rtl" style="text-align:right">

---
# بخش ۱: عملیات کانولوشن

**کانولوشن** یک شبکه کوچک از وزن‌ها، به نام **کرنل** (یا فیلتر)، را روی تصویر می‌لغزاند. در هر موقعیت، خروجی جمع وزن‌دار تکه تصویر زیر کرنل است؛ برای کرنل ۳×۳:

$$\text{out}(i, j) = \sum_{a=1}^{3} \sum_{b=1}^{3} \; \text{kernel}(a, b) \cdot \text{image}(i+a, j+b)$$

این همان جمع وزن‌داری است که همه مدل‌های این دوره حساب می‌کنند، با دو محدودیت عمدی:

- **محلی بودن.** هر مقدار خروجی فقط به یک همسایگی کوچک وابسته است، هماهنگ با ساختار تصویر: پیکسل‌های نزدیک مرتبط‌اند و دورها عمدتا نه.
- **وزن مشترک.** *همان* چند وزن در همه موقعیت‌ها اعمال می‌شود، پس هر الگویی که کرنل آشکار کند، همه‌جای تصویر آشکارش می‌کند. در مقابل، لایه تمام‌متصل هر وزن را به یک مکان مطلق پیکسل گره می‌زند.

پیاده‌سازی، رونویسی مستقیم فرمول است:

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def convolve(image, kernel):
    kh, kw = kernel.shape
    H, W = image.shape
    out = np.zeros((H - kh + 1, W - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = (image[i:i+kh, j:j+kw] * kernel).sum()
    return out

print("کانولوشن پیاده‌سازی شد ✅")

<div dir="rtl" style="text-align:right">

کار یک کرنل را بهتر از همه روی عکس واقعی می‌شود دید (چند عکس نمونه با مجوز آزاد در پوشه `images` کنار همین نوت‌بوک قرار دارد؛ هر فایلی که نباشد خودکار از مخزن دوره دانلود می‌شود). کرنل زیر برای هر تکه ۳×۳، ستون چپ منهای ستون راست را حساب می‌کند؛ پس دقیقا جایی پاسخ می‌دهد که روشنایی در جهت افقی تغییر کند، یعنی یک **لبه عمودی**:

</div>

In [ ]:
import os
from urllib.request import urlretrieve

BASE_URL = "https://raw.githubusercontent.com/SharifiZarchi/IntroAI/main/Session_08/ComputerVision/images/"

def load_image(name):
    if not os.path.exists(os.path.join("images", name)):
        os.makedirs("images", exist_ok=True)
        urlretrieve(BASE_URL + name, os.path.join("images", name))
    return plt.imread(os.path.join("images", name))     # آرایه float در بازه [0, 1]

image = load_image("camera.png")
print("ابعاد تصویر:", image.shape)

vertical_edge = np.array([[1.0, 0.0, -1.0],
                          [1.0, 0.0, -1.0],
                          [1.0, 0.0, -1.0]])

feature_map = convolve(image, vertical_edge)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))
ax1.imshow(image, cmap="gray"); ax1.set_title("input"); ax1.axis("off")
ax2.imshow(np.abs(feature_map), cmap="gray"); ax2.set_title("output of the vertical-edge kernel"); ax2.axis("off")
plt.show()

<div dir="rtl" style="text-align:right">

خروجی، که **نقشه ویژگی (feature map)** نام دارد، هر جا الگو رخ داده روشن می‌شود، در هر موقعیتی، با همان یک مجموعه ۹تایی وزن. کرنل‌های مختلف ساختارهای مختلفی استخراج می‌کنند؛ مجموعه کوچکی از کرنل‌ها از همین حالا تصویر را خوب خلاصه می‌کند. دو تمرین همین شهود را می‌سازند؛ بعد بخش ۲ طراحی کرنل را به گرادیان کاهشی می‌سپارد.

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۱
کرنلی طراحی کن که لبه‌های **افقی** را آشکار کند، روی همان تصویر اعمالش کن و نتیجه را نمایش بده. (چه نسبتی با `vertical_edge` دارد؟)

</div>

In [ ]:
# ✏️ horizontal_edge = ...



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
horizontal_edge = vertical_edge.T      # ترانهاده: سطر بالا منهای سطر پایین

fm = convolve(image, horizontal_edge)
plt.figure(figsize=(5.5, 5.5))
plt.imshow(np.abs(fm), cmap="gray"); plt.axis("off")
plt.title("horizontal edges")
plt.show()
```

ترانهاده نقش سطر و ستون را عوض می‌کند، پس آشکارساز به تغییر روشنایی عمودی پاسخ می‌دهد، یعنی لبه‌های افقی (خط افق، لبه بالای دوربین). کرنل‌ها آشکارسازهای جهت‌دار الگو هستند.

</details>

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۲
هر کرنلی آشکارساز لبه نیست. این دو را اعمال کن و هر خروجی را در یک جمله توصیف کن:

- کرنل **میانگین‌گیر**: شبکه ۳×۳ از $1/9$
- کرنل **تیزکننده**: $\begin{pmatrix} 0 & -1 & 0 \\ -1 & 5 & -1 \\ 0 & -1 & 0 \end{pmatrix}$

</div>

In [ ]:
# ✏️ blur = ...
# ✏️ sharpen = ...



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
blur = np.ones((3, 3)) / 9
sharpen = np.array([[0., -1., 0.], [-1., 5., -1.], [0., -1., 0.]])

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, (title, k) in zip(axes, [("input", None), ("blur", blur), ("sharpen", sharpen)]):
    ax.imshow(image if k is None else convolve(image, k).clip(0, 1), cmap="gray")
    ax.set_title(title); ax.axis("off")
plt.show()
```

میانگین‌گیری هر پیکسل را با میانگین همسایگی‌اش جایگزین می‌کند و جزئیات ریز را صاف می‌کند (دقیقا همان «محو» در ویرایشگرهای عکس). کرنل تیزکننده اختلاف هر پیکسل با همسایه‌ها را تشدید می‌کند و جزئیات را اغراق می‌کند. همه فیلترهای کلاسیک تصویر کانولوشن‌اند؛ تفاوت فیلترها فقط در آن ۹ عدد است.

</details>

</div>

<div dir="rtl" style="text-align:right">

---
# بخش ۲: از کرنل تا لایه

‏CNN سه نوع لایه را روی هم می‌چیند:

- **`nn.Conv2d(in, out, 3)`**: لایه کانولوشن با `out` کرنل یادگرفتنی که هر کدام همه `in` کانال ورودی را می‌پوشانند؛ خروجی برای هر کرنل یک نقشه ویژگی است. کرنل‌ها تصادفی شروع می‌شوند و با پس‌انتشار آموزش می‌بینند، مثل هر وزن دیگری از جلسه ۷ به بعد. هیچ‌کس آن‌ها را با دست طراحی نمی‌کند.
- **`nn.ReLU()`**: فعال‌سازی همیشگی، عنصربه‌عنصر روی نقشه‌های ویژگی.
- **`nn.MaxPool2d(2)`**: هر بلوک ۲×۲ را با بیشینه‌اش جایگزین می‌کند و طول و عرض را نصف. محاسبات پایین‌دستی چهار برابر ارزان‌تر می‌شود و نمایش نسبت به جابه‌جایی‌های یکی‌دو پیکسلی تحمل پیدا می‌کند، چون بیشینه بلوک از آن‌ها جان به در می‌برد.

چیدن بلوک‌های (کانولوشن، ReLU، ادغام) روی هم، **سلسله‌مراتب ویژگی** می‌سازد: کرنل‌های بلوک اول پیکسل خام می‌بینند و آشکارساز لبه و رنگ می‌شوند؛ بلوک بعدی نقشه‌های ادغام‌شده را می‌بیند، پس کرنل‌هایش به *ترکیب* لبه‌ها در پهنه وسیع‌تری پاسخ می‌دهند؛ و همین‌طور ادامه دارد. این همان ایده انتزاع لایه‌به‌لایه اسلایدهای جلسه ۷ است، این بار به‌طور ملموس.

اقتصاد پارامترها چشمگیر است و یک بار حساب‌کردنش با دست می‌ارزد. لایه تمام‌متصل از تصویر ۳×۳۲×۳۲ به ۲۵۶ واحد، ‎۲۵۶ × ۳۰۷۲ ≈ ۷۸۶هزار وزن می‌خواهد. لایه `Conv2d(3, 16, 3)`، ‏۱۶ کرنل × (۳ × ۳ × ۳) وزن + ۱۶ بایاس = **۴۴۸ پارامتر**، مستقل از اندازه تصویر. همین عدد، بررسی‌شده در کد:

</div>

In [ ]:
import torch
from torch import nn

layer = nn.Conv2d(3, 16, kernel_size=3)
print("پارامترهای Conv2d(3, 16, 3):", sum(p.numel() for p in layer.parameters()))

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۳
تعداد پارامترهای `nn.Conv2d(16, 32, kernel_size=3)` را با دست حساب کن (۱۶ کانال ورودی، ۳۲ کرنل ۳×۳)، بعد با PyTorch بررسی کن.

</div>

In [ ]:
layer2 = nn.Conv2d(16, 32, kernel_size=3)
# ✏️ اول محاسبه دستی؛ بعد:
# print(sum(p.numel() for p in layer2.parameters()))



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
print(32 * (16 * 3 * 3) + 32)                          # 4640
print(sum(p.numel() for p in layer2.parameters()))     # 4640
```

هر کدام از ۳۲ کرنل همه ۱۶ کانال ورودی را می‌پوشاند: ‎۱۶ × ۳ × ۳ = ۱۴۴ وزن به‌علاوه یک بایاس، و ‎۳۲ × ۱۴۵ = ۴۶۴۰، مستقل از اندازه تصویر. همین استقلال است که یک معماری را از تصویر بندانگشتی تا عکس مگاپیکسلی مقیاس‌پذیر می‌کند.

</details>

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۴
دنبال‌کردن ابعاد، مهارت روزمره مهندسی CNN. ورودی‌ای با ابعاد `(3, 32, 32)` از این‌ها می‌گذرد:

`Conv2d(3, 16, 3, padding=1)` ← `MaxPool2d(2)` ← `Conv2d(16, 32, 3, padding=1)` ← `MaxPool2d(2)` ← `Flatten()`

ابعاد بعد از هر مرحله را روی کاغذ به دست بیاور (`padding=1` طول و عرض را در کانولوشن ۳×۳ ثابت نگه می‌دارد؛ ادغام نصفشان می‌کند). ‏`Flatten` چه طولی تولید می‌کند؟ با عبوردادن یک تنسور ساختگی از لایه‌ها، مرحله‌به‌مرحله بررسی کن.

</div>

In [ ]:
x = torch.zeros(1, 3, 32, 32)
# ✏️ لایه‌ها را مرحله‌به‌مرحله اعمال کن و بعد از هر کدام x.shape را چاپ کن



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
steps = [nn.Conv2d(3, 16, 3, padding=1), nn.MaxPool2d(2),
         nn.Conv2d(16, 32, 3, padding=1), nn.MaxPool2d(2), nn.Flatten()]
x = torch.zeros(1, 3, 32, 32)
for step in steps:
    x = step(x)
    print(step.__class__.__name__, tuple(x.shape))
```

‏`(16, 32, 32)` ← `(16, 16, 16)` ← `(32, 16, 16)` ← `(32, 8, 8)` ← `2048`. طول تخت‌شده ‎۳۲ × ۸ × ۸ = ۲۰۴۸ همان جایی است که اندازه ورودی اولین لایه خطی مدل بخش ۳ از آن می‌آید؛ ناسازگاری همین عدد رایج‌ترین باگ CNN است.

</details>

</div>

<div dir="rtl" style="text-align:right">

---
# بخش ۳: یک CNN روی CIFAR-10، در برابر شبکه تمام‌متصل

محک جلسه ۷: شبکه تمام‌متصل با ۷۸۹٬۲۵۸ پارامتر به دقت آزمون **۰.۴۹۹** روی CIFAR-10 رسید (Adam، ۱۰ ایپاک). همان داده، همان بهینه‌ساز، همان ۱۰ ایپاک، معماری پیچشی:

</div>

In [ ]:
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms

train_data = torchvision.datasets.CIFAR10(root="data", train=True, download=True,
                                          transform=transforms.ToTensor())
test_data = torchvision.datasets.CIFAR10(root="data", train=False, download=True,
                                         transform=transforms.ToTensor())
print("آموزش:", len(train_data), "| آزمون:", len(test_data))

In [ ]:
torch.manual_seed(42)

cnn = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=1),   # ۳ کانال رنگ ← ۱۶ نقشه ویژگی
    nn.ReLU(),
    nn.MaxPool2d(2),                              # ۳۲×۳۲ ← ۱۶×۱۶
    nn.Conv2d(16, 32, kernel_size=3, padding=1),  # ۱۶ نقشه ← ۳۲ نقشه
    nn.ReLU(),
    nn.MaxPool2d(2),                              # ۱۶×۱۶ ← ۸×۸
    nn.Flatten(),
    nn.Linear(32 * 8 * 8, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

print("پارامترها:", sum(p.numel() for p in cnn.parameters()))

<div dir="rtl" style="text-align:right">

‏۲۶۸٬۶۵۰ پارامتر، حدود یک‌سوم مدل تمام‌متصل. حلقه آموزش بدون تغییر از بخش ۸ جلسه ۷ است؛ فقط `model` فرق دارد:

</div>

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("پردازنده:", device)

cnn = cnn.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)

for epoch in range(10):
    total = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = loss_fn(cnn(images), labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * len(labels)
    print(f"epoch {epoch + 1:2d} | mean loss {total / len(train_data):.3f}")

In [ ]:
test_loader = DataLoader(test_data, batch_size=512)

def accuracy(model):
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            correct += (model(images.to(device)).argmax(dim=1).cpu() == labels).sum().item()
    return correct / len(test_data)

print("دقت آزمون CNN:", round(accuracy(cnn), 3))

<div dir="rtl" style="text-align:right">

نتیجه رودررو:

| | تمام‌متصل (جلسه ۷) | CNN (همین بخش) |
|---|---|---|
| پارامترها | ۷۸۹٬۲۵۸ | **۲۶۸٬۶۵۰** |
| دقت آزمون CIFAR-10 | ۰.۴۹۹ | **≈ ۰.۶۹** |
| وزن به ازای هر الگو | یکی برای هر مکان مطلق پیکسل | یک کرنل کوچک، مشترک همه‌جا |

سه برابر پارامتر کمتر و بیست واحد دقت بیشتر، فقط از معماری: محلی بودن و وزن مشترک همان ساختاری از تصویر را در مدل تعبیه می‌کنند که لایه تمام‌متصل باید از داده کشفش می‌کرد. تطبیق معماری با ساختار داده اصل مرکزی طراحی در یادگیری عمیق است؛ ترنسفورمرهای جلسه ۱۰ همین اصل‌اند برای زبان.

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۵
دقت به تفکیک کلاس CNN را حساب کن و با عددهای شبکه تمام‌متصل جلسه ۷ مقایسه کن: هواپیما ۰.۳۴، خودرو ۰.۶۱، پرنده ۰.۳۴، گربه ۰.۳۹، گوزن ۰.۳۷، سگ ۰.۴۵، قورباغه ۰.۶۴، اسب ۰.۴۶، کشتی ۰.۷۲، کامیون ۰.۶۲. کدام کلاس‌ها بیشترین رشد را کرده‌اند و چه اشتراکی دارند؟

</div>

In [ ]:
# ✏️ دقت به تفکیک کلاس برای CNN



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        all_preds.append(cnn(images.to(device)).argmax(dim=1).cpu())
        all_labels.append(labels)
preds = torch.cat(all_preds); labels = torch.cat(all_labels)

for c, name in enumerate(test_data.classes):
    mask = labels == c
    print(f"{name:12s} {(preds[mask] == c).float().mean():.2f}")
```

همه کلاس‌ها بهتر می‌شوند؛ بزرگ‌ترین جهش‌ها هواپیما (۰.۳۴ به حدود ۰.۸۱) و پرنده (۰.۳۴ به حدود ۰.۶۰) هستند، کلاس‌هایی که سوژه‌شان در مکان‌ها و مقیاس‌های بسیار متغیر ظاهر می‌شود. آشکارساز مستقل از مکان دقیقا همان‌جایی بیشترین کمک را می‌کند که مکان بیشترین تغییر را داشت.

</details>

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۶
وزن مشترک پیش‌بینی می‌کند که CNN باید جابه‌جایی‌های کوچک ورودی را تحمل کند. پیش‌بینی را بیازما: ‏CNN آموزش‌دیده را روی تصاویر آزمون ۲ و ۴ پیکسل جابه‌جاشده به راست ارزیابی کن (`torch.roll(images, shifts=k, dims=3)` داخل حلقه ارزیابی) و دقت‌ها را گزارش بده.

</div>

In [ ]:
# ✏️ ارزیابی با ورودی جابه‌جاشده



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
def accuracy_shifted(model, k):
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = torch.roll(images, shifts=k, dims=3)
            correct += (model(images.to(device)).argmax(dim=1).cpu() == labels).sum().item()
    return correct / len(test_data)

for k in (0, 2, 4):
    print(f"shift {k}px: {accuracy_shifted(cnn, k):.3f}")
```

حدودا ۰.۶۹ ← ۰.۶۶ ← ۰.۶۰: افتی ملایم (که بخشی‌اش هم از این است که `roll` پیکسل‌ها را دور می‌زند و از لبه مقابل وارد می‌کند). کرنل مشترک الگو را هر جا فرود بیاید می‌بیند و ادغام، ناهم‌ترازی باقی‌مانده را جذب می‌کند. شبکه تمام‌متصل، که هر وزنش به یک مکان پیکسل بسته است، چنین سازوکاری ندارد.

</details>

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۷
مدل را با بلوک سوم گسترش بده: `Conv2d(32, 64, 3, padding=1)` + ReLU + ادغام؛ اندازه‌های `Flatten`/`Linear` را با روش تمرین ۴ اصلاح کن، بازآموزی کن و پارامترها و دقت را گزارش بده. هر بار یک تغییر، با اندازه‌گیری؛ کل هنر کار معماری همین است.

</div>

In [ ]:
# ✏️ CNN سه‌بلوکه



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
torch.manual_seed(42)
cnn3 = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # ← ۱۶×۱۶×۱۶
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # ← ۳۲×۸×۸
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # ← ۶۴×۴×۴
    nn.Flatten(),
    nn.Linear(64 * 4 * 4, 128), nn.ReLU(), nn.Linear(128, 10),
).to(device)
print("پارامترها:", sum(p.numel() for p in cnn3.parameters()))
# با همان حلقه بازآموزی کن، بعد: print(round(accuracy(cnn3), 3))
```

بلوک سوم سلسله‌مراتب را عمیق‌تر می‌کند و ادغام اضافه، طول تخت‌شده (‎۶۴ × ۴ × ۴ = ۱۰۲۴) را از قبل هم *کوچک‌تر* می‌کند؛ پس مدل می‌تواند با پارامتر کمتر، معمولا چند واحد دقت بیشتر بگیرد. اندازه بگیر؛ هرگز فرض نکن تغییر معماری کمک کرده است.

</details>

</div>

<div dir="rtl" style="text-align:right">

---
# بخش ۴: درون CNN آموزش‌دیده

### ۴.۱ کرنل‌های یادگرفته‌شده
‏۱۶ کرنل لایه اول ۳×۳×۳ هستند؛ آن‌قدر کوچک که به‌صورت وصله‌های رنگی نمایش دادنی‌اند (برای نمایش نرمال شده‌اند):

</div>

In [ ]:
kernels = cnn[0].weight.detach().cpu()
kernels = (kernels - kernels.min()) / (kernels.max() - kernels.min())

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for j, ax in enumerate(axes.ravel()):
    ax.imshow(kernels[j].permute(1, 2, 0))
    ax.axis("off")
plt.suptitle("the 16 learned first-layer kernels (3×3, in color)")
plt.tight_layout(); plt.show()

<div dir="rtl" style="text-align:right">

در وضوح ۳×۳، نمای آموزنده کاری است که *می‌کنند*. عبور یک تصویر آزمون از کانولوشن اول و ReLU شانزده نقشه ویژگی تولید می‌کند:

</div>

In [ ]:
image, label = test_data[0]

with torch.no_grad():
    maps = nn.Sequential(cnn[0], cnn[1])(image.unsqueeze(0).to(device))[0].cpu()

fig, axes = plt.subplots(2, 9, figsize=(13, 3.2))
axes[0, 0].imshow(image.permute(1, 2, 0)); axes[0, 0].set_title(test_data.classes[label], fontsize=8)
axes[1, 0].axis("off")
for j in range(8):
    axes[0, j + 1].imshow(maps[j], cmap="gray")
    axes[1, j + 1].imshow(maps[j + 8], cmap="gray")
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("input and its 16 first-layer feature maps")
plt.tight_layout(); plt.show()

<div dir="rtl" style="text-align:right">

نقشه‌های مختلف به ساختارهای مختلف پاسخ می‌دهند: لبه‌های جهت‌دار، ناحیه‌های رنگی. این‌ها همان جنس آشکارسازهای دست‌ساز بخش ۱ هستند، جز این‌که شبکه **خودش انتخابشان کرده**، با گرادیان کاهشی، چون به دسته‌بندی کمک می‌کنند. با تصاویر وزن لایه متراکم جلسه ۷ مقایسه کن که ساختار خواندنی نداشتند: با دادن واژگان مکانی به مدل، آشکارسازهای تفسیرپذیر حتی در شبکه‌ای کوچک هم پدید می‌آیند.

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۸
نقشه‌های ویژگی را بعد از بلوک دوم کانولوشن نمایش بده (لایه‌های `cnn[0]` تا `cnn[4]`)؛ ابعادشان ۱۶×۱۶ است و ۳۲ تا هستند؛ ۱۶ تای اول را نشان بده. شخصیتشان چه فرقی با نقشه‌های لایه اول دارد؟

</div>

In [ ]:
# ✏️ نقشه‌های ویژگی بلوک دوم



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
with torch.no_grad():
    maps2 = nn.Sequential(*cnn[:5])(image.unsqueeze(0).to(device))[0].cpu()
print(maps2.shape)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for j, ax in enumerate(axes.ravel()):
    ax.imshow(maps2[j], cmap="gray")
    ax.axis("off")
plt.suptitle("16 of the 32 second-layer feature maps (16×16)")
plt.tight_layout(); plt.show()
```

درشت‌تر و انتزاعی‌تر: هر مقدار تکه بزرگ‌تری از تصویر اصلی را خلاصه می‌کند و نقشه‌ها به ترکیب الگوهای لایه اول پاسخ می‌دهند نه به لبه‌های خام. سلسله‌مراتب، مشاهده‌شده به‌طور مستقیم.

</details>

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۹
برای همان تصویر آزمون، پیدا کن کدام نقشه ویژگی لایه اول به‌طور میانگین قوی‌تر از همه فعال شده (`maps[j].mean()`)؛ آن نقشه را کنار ورودی نمایش بده و توصیف کن کرنلش ظاهرا چه چیزی را آشکار می‌کند.

</div>

In [ ]:
# ✏️ قوی‌ترین نقشه ویژگی



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
j = maps.mean(dim=(1, 2)).argmax().item()
print("قوی‌ترین نقشه:", j)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3.5))
ax1.imshow(image.permute(1, 2, 0)); ax1.set_title("input"); ax1.axis("off")
ax2.imshow(maps[j], cmap="gray"); ax2.set_title(f"feature map {j}"); ax2.axis("off")
plt.show()
```

این‌که کدام نقشه برنده شود به محتوای تصویر بستگی دارد (برای خیلی از تصاویر، نقشه‌ای که به رنگ غالب یا یک کانتور قوی پاسخ می‌دهد). خواندن فعال‌سازی‌های درونی مدل در برابر ورودی‌اش، دقیقا مثل همین‌جا، ابزار پایه اشکال‌زدایی و تفسیرپذیری CNN است.

</details>

</div>

<div dir="rtl" style="text-align:right">

---
# بخش ۵: داده‌افزایی (data augmentation)

برای مدل نباید فرقی کند که گربه رو به چپ است یا راست، کمی خارج از مرکز نشسته یا عکس کمی تاریک‌تر گرفته شده. **داده‌افزایی** همین را در آموزش تعبیه می‌کند: هر بار که تصویری استفاده می‌شود، اول یک تبدیل تصادفی حافظ برچسب (قرینه، جابه‌جایی یا چرخش کوچک، تغییر روشنایی) رویش اعمال می‌شود؛ پس شبکه در هر ایپاک نسخه تازه‌ای می‌بیند و نمی‌تواند پیکسل‌های دقیق را حفظ کند. این ابزار جزء استاندارد خط لوله‌های واقعی آموزش است و یکی از پاسخ‌های عملی به بیش‌برازش (جلسه ۳).

کتابخانه استاندارد **albumentations** است (در Colab از قبل نصب است؛ برای اجرای محلی یک بار `pip install albumentations`). خط لوله فهرستی از تبدیل‌هاست، هر کدام با احتمال خودش؛ صدازدنش روی یک تصویر، یک قرعه تصادفی اعمال می‌کند:

</div>

In [ ]:
import albumentations as A

augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(translate_percent=0.1, scale=(0.9, 1.1), rotate=(-15, 15), p=1.0),
    A.RandomBrightnessContrast(p=0.5),
])

cat = (load_image("chelsea.png") * 255).astype(np.uint8)   # ورودی albumentations باید uint8 باشد

fig, axes = plt.subplots(2, 4, figsize=(13, 5.5))
axes[0, 0].imshow(cat); axes[0, 0].set_title("original", fontsize=9)
for ax in axes.ravel()[1:]:
    ax.imshow(augment(image=cat)["image"])
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("one photo, seven random augmentations")
plt.tight_layout(); plt.show()

<div dir="rtl" style="text-align:right">

همان گربه، همان برچسب، و در هر قرعه پیکسل‌های متفاوت. همان خط لوله روی یک نمونه CIFAR-10، در وضوحی که شبکه واقعا می‌بیند:

</div>

In [ ]:
sample = train_data.data[7]                  # numpy با ابعاد 32x32x3 و نوع uint8

fig, axes = plt.subplots(1, 8, figsize=(13, 2))
axes[0].imshow(sample); axes[0].set_title("original", fontsize=8)
for ax in axes[1:]:
    ax.imshow(augment(image=sample)["image"])
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

<div dir="rtl" style="text-align:right">

وصل‌کردن داده‌افزایی به آموزش یک قدم است: خط لوله را داخل دیتاست اعمال کن تا هر بچ تازه ساخته شود:

</div>

In [ ]:
class AugmentedCIFAR(torch.utils.data.Dataset):
    def __init__(self, base, pipeline):
        self.base = base
        self.pipeline = pipeline
    def __len__(self):
        return len(self.base)
    def __getitem__(self, i):
        img = self.pipeline(image=self.base.data[i])["image"]
        return torch.tensor(img).permute(2, 0, 1).float() / 255.0, self.base.targets[i]

augmented_loader = DataLoader(AugmentedCIFAR(train_data, augment), batch_size=128, shuffle=True)

images, labels = next(iter(augmented_loader))
print("یک بچ، تازه داده‌افزایی‌شده:", tuple(images.shape))

<div dir="rtl" style="text-align:right">

دادن `augmented_loader` به حلقه آموزش بخش ۳ تمام کاری است که لازم است. سود آن وقتی ظاهر می‌شود که مدل شروع به بیش‌برازش کند: شبکه دیگر هرگز یک تصویر را دو بار عینا نمی‌بیند، پس حفظ‌کردن گزینه نیست.

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۱۰
به یک نسخه از خط لوله `A.VerticalFlip(p=1.0)` اضافه کن و چند نمونه CIFAR را نمایش بده. تبدیل از نظر فنی معتبر است؛ چرا برای این مجموعه‌داده انتخاب بدی است؟ (کامیون وارونه چه چیزی به شبکه یاد می‌دهد؟)

</div>

In [ ]:
# ✏️ خط لوله با قرینه عمودی و نگاهی به خروجی‌اش



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
flipped = A.Compose([A.VerticalFlip(p=1.0)])

fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, i in zip(axes, range(6)):
    ax.imshow(flipped(image=train_data.data[i])["image"])
    ax.axis("off")
plt.show()
```

کامیون، گوزن و کشتی وارونه در عکس‌های واقعی عملا هرگز رخ نمی‌دهند؛ پس شبکه ظرفیتش را صرف ورودی‌هایی می‌کند که در آزمون هیچ‌وقت نمی‌بیند. داده‌افزایی باید داخل توزیع واقعی داده بماند؛ انتخاب تبدیل‌ها دانش دامنه است. (برای تصاویر ماهواره‌ای یا میکروسکوپی که هر چرخشی طبیعی است، انواع چرخش و قرینه دقیقا درست‌اند.)

</details>

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۱۱
خط لوله‌ای از خودت با دو تبدیل استفاده‌نشده در بالا بساز، مثلا `A.GaussianBlur` و `A.CoarseDropout`، و هشت نسخه از عکس گربه را نمایش بده. مستندات albumentations ده‌ها گزینه دارد.

</div>

In [ ]:
# ✏️ خط لوله تو، هشت نسخه



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
mine = A.Compose([
    A.GaussianBlur(p=0.5),
    A.CoarseDropout(num_holes_range=(1, 3), hole_height_range=(20, 60),
                    hole_width_range=(20, 60), p=1.0),
])

fig, axes = plt.subplots(2, 4, figsize=(13, 5.5))
for ax in axes.ravel():
    ax.imshow(mine(image=cat)["image"])
    ax.axis("off")
plt.show()
```

‏`CoarseDropout` مستطیل‌های تصادفی را حذف می‌کند و شبکه را وادار می‌کند به جای یک تکه لودهنده، از کل شیء استفاده کند؛ محوشدگی هم تغییر فوکوس را شبیه‌سازی می‌کند. هر دو در خط لوله‌های صنعتی رایج‌اند.

</details>

</div>

<div dir="rtl" style="text-align:right">

---
# بخش ۶: یادگیری انتقالی با ResNet18

مقیاس‌دادن به دستور بخش ۳ (بلوک بیشتر، داده بیشتر، محاسبات بیشتر) همان چیزی است که بینایی را در صنعت حل کرد. **ResNet18** یک CNN استاندارد ۱۸ لایه با ۱۱.۷ میلیون پارامتر است، آموزش‌دیده روی ImageNet (‏۱.۲ میلیون عکس، ۱۰۰۰ دسته)؛ `torchvision` وزن‌های آموزش‌دیده را تحویل می‌دهد (~۴۵ مگابایت، یک بار دانلود). همین‌طور آماده، عکس دسته‌بندی می‌کند:

</div>

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT
resnet = resnet18(weights=weights).eval()
preprocess = weights.transforms()          # تغییر اندازه و نرمال‌سازی موردانتظار ResNet
categories = weights.meta["categories"]

for name, img in [("chelsea", load_image("chelsea.png")), ("coffee", load_image("coffee.png"))]:
    x = preprocess(torch.tensor(img).permute(2, 0, 1)).unsqueeze(0)
    with torch.no_grad():
        probs = resnet(x).softmax(dim=1)[0]
    top = probs.topk(3)
    print(name, "→", [(categories[i], round(v.item(), 2)) for v, i in zip(top.values, top.indices)])

<div dir="rtl" style="text-align:right">

### ۶.۱ روال حرفه‌ای: بازاستفاده از ویژگی‌ها
دیگر تقریبا هیچ‌کس مدل بینایی را از صفر آموزش نمی‌دهد. حرکت استاندارد، **یادگیری انتقالی**، پشته کانولوشنی شبکه ازپیش‌آموزش‌دیده را استخراج‌کننده ویژگی عمومی می‌گیرد و فقط یک دسته‌بند تازه را روی داده خود آموزش می‌دهد.

آزمایش زیر ادعا را کمی می‌کند. لایه آخر ResNet18 را با همانی جایگزین کن تا شبکه بردار ویژگی ۵۱۲بعدی‌اش را خروجی بدهد؛ **۵۰۰۰** تصویر آموزش CIFAR-10 (یک‌دهم مجموعه) را از آن عبور بده؛ و یک رگرسیون لجستیک روی آن ویژگی‌ها آموزش بده. هیچ وزن کانولوشنی‌ای به‌روزرسانی نمی‌شود. (دو سلول استخراج، سلول‌های کند نوت‌بوک‌اند، مجموعا حدود ۲ دقیقه؛ ارزیابی با ۲۰۰۰ تصویر آزمون.)

</div>

In [ ]:
from torch.utils.data import Subset

backbone = resnet18(weights=weights)
backbone.fc = nn.Identity()               # حذف سر ۱۰۰۰کلاسه: خروجی = ویژگی ۵۱۲بعدی
backbone.eval()

def extract_features(dataset, n):
    features, labels = [], []
    with torch.no_grad():
        for images, ys in DataLoader(Subset(dataset, range(n)), batch_size=64):
            features.append(backbone(preprocess(images)))
            labels.append(ys)
    return torch.cat(features).numpy(), torch.cat(labels).numpy()

F_train, y_train = extract_features(train_data, 5000)
F_test, y_test = extract_features(test_data, 2000)
print("ماتریس‌های ویژگی:", F_train.shape, F_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

probe = LogisticRegression(max_iter=2000)
probe.fit(F_train, y_train)

print("دقت انتقالی (با ۵۰۰۰ تصویر آموزش):", round(probe.score(F_test, y_test), 3))

<div dir="rtl" style="text-align:right">

جدول امتیازات این جلسه تا این‌جا:

| مدل | تصاویر آموزش | پارامترهای آموزش‌دیده | دقت CIFAR-10 |
|---|---|---|---|
| تمام‌متصل (جلسه ۷) | ۵۰هزار | ۷۸۹هزار | ۰.۴۹۹ |
| CNN از صفر (بخش ۳) | ۵۰هزار | ۲۶۹هزار | ≈ ۰.۶۹ |
| **ویژگی‌های ResNet18 + دسته‌بند خطی** | **۵هزار** | **۵هزار (فقط دسته‌بند)** | **≈ ۰.۸۲** |

یک مدل خطی روی ویژگی‌های ازپیش‌آموخته، با **یک‌دهم داده** و بدون به‌روزرسانی هیچ وزن کانولوشنی، از CNNای که روی همه‌چیز آموزش دادیم بهتر می‌شود. ویژگی‌هایی که ResNet از ۱.۲ میلیون عکس آموخته به مجموعه‌داده‌ای که هرگز ندیده منتقل می‌شوند؛ به همین دلیل یادگیری انتقالی پیش‌فرض کار حرفه‌ای است و «چقدر داده برچسب‌دار داری» دیگر اولین مانع بینایی کاربردی نیست.

</div>

<div dir="rtl" style="text-align:right">

### ۶.۲ ریزتنظیم واقعی (fine-tuning)
‏probe ستون‌فقرات را منجمد نگه داشت. **ریزتنظیم** جلوتر می‌رود: سر شبکه را با یک لایه تازه ۱۰کلاسه جایگزین کن و *همه* وزن‌ها را روی داده جدید به‌روزرسانی کن، با نرخ یادگیری کوچک تا ویژگی‌های ازپیش‌آموخته تنظیم شوند نه نابود. روی GPU T4 در Colab دو ایپاک زیر حدود یک دقیقه است؛ روی CPU حدود ده دقیقه:

</div>

In [ ]:
torch.manual_seed(42)
finetuned = resnet18(weights=weights)
finetuned.fc = nn.Linear(512, 10)                 # سر جدید برای ۱۰ کلاس ما
finetuned = finetuned.to(device)

optimizer = torch.optim.Adam(finetuned.parameters(), lr=1e-4)   # کوچک: تنظیم کن، نابود نکن
train_5k = DataLoader(Subset(train_data, range(5000)), batch_size=32, shuffle=True)

for epoch in range(2):
    finetuned.train()
    total = 0.0
    for images, labels in train_5k:
        images, labels = images.to(device), labels.to(device)
        loss = loss_fn(finetuned(preprocess(images)), labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * len(labels)
    print(f"epoch {epoch + 1} | mean loss {total / 5000:.3f}")

finetuned.eval()
correct = 0
with torch.no_grad():
    for images, labels in DataLoader(Subset(test_data, range(2000)), batch_size=64):
        images = images.to(device)
        correct += (finetuned(preprocess(images)).argmax(dim=1).cpu() == labels).sum().item()
print("دقت پس از ریزتنظیم:", round(correct / 2000, 3))

<div dir="rtl" style="text-align:right">

جدول امتیازات، کامل:

| مدل | تصاویر آموزش | پارامترهای آموزش‌دیده | دقت CIFAR-10 |
|---|---|---|---|
| تمام‌متصل (جلسه ۷) | ۵۰هزار | ۷۸۹هزار | ۰.۴۹۹ |
| CNN از صفر (بخش ۳) | ۵۰هزار | ۲۶۹هزار | ≈ ۰.۶۹ |
| ویژگی‌های ResNet18 + دسته‌بند خطی | ۵هزار | ۵هزار | ≈ ۰.۸۲ |
| **ResNet18 ریزتنظیم‌شده** | **۵هزار** | **۱۱.۷ میلیون** | **≈ ۰.۸۷** |

تطبیق‌دادن ویژگی‌های ازپیش‌آموخته با وظیفه، از منجمدکردنشان بهتر است، به بهای محاسبات بیشتر. همین دو گزینه، ویژگی منجمد برای سرعت و ریزتنظیم برای دقت، منوی استاندارد بینایی ماشین کاربردی است و هر دو همین‌جا با یک‌دهم مجموعه‌داده اجرا شدند.

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۱۲
بازده داده: رگرسیون لجستیک را فقط روی **۱۰۰۰** سطر اول `F_train` بازآموزی و ارزیابی کن. هر چهار عدد را مقایسه کن (تمام‌متصل، CNN از صفر، probe با ۵هزار، probe با ۱هزار). داده برچسب‌دار کجا دیگر گلوگاه نیست؟

</div>

In [ ]:
# ✏️ probe با ۱۰۰۰ تصویر



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
probe_small = LogisticRegression(max_iter=2000)
probe_small.fit(F_train[:1000], y_train[:1000])
print(round(probe_small.score(F_test, y_test), 3))
```

حدود ۰.۷۹: با **یک‌پنجاهم** برچسب‌ها، خط لوله ویژگی‌های ازپیش‌آموخته هنوز به‌وضوح از هر دو شبکه از-صفر که روی کل ۵۰هزار تصویر آموزش دیدند بهتر است. پیش‌آموزش، گلوگاه را از داده برچسب‌دار به کیفیت ویژگی‌ها منتقل کرد؛ دقیقا همان انتقالی که بعدها مدل‌های زبانی بزرگ را عملی کرد (جلسه ۱۰).

</details>

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۱۳
‏`load_image("astronaut.png")` (پرتره فضانورد آیلین کالینز) را با دسته‌بند ResNet دسته‌بندی کن و ۵ برچسب برتر را چاپ کن. نتیجه خراب به نظر می‌رسد. در `categories` جست‌وجو کن و دقیقا بگو چه چیزی غایب است. نتیجه‌گیری‌ات را نگه دار؛ بخش ۷ به همین تصویر برمی‌گردد.

</div>

In [ ]:
# ✏️ فضانورد، ۵ برچسب برتر



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
img = load_image("astronaut.png")
x = preprocess(torch.tensor(img).permute(2, 0, 1)).unsqueeze(0)
with torch.no_grad():
    probs = resnet(x).softmax(dim=1)[0]
top = probs.topk(5)
for value, idx in zip(top.values, top.indices):
    print(f"{categories[idx]:20s} {value.item():.2f}")

print("person" in categories)      # False
```

حدس‌های کم‌اطمینان درباره اشیای پس‌زمینه، چون **در ۱۰۰۰ دسته ImageNet کلاس «انسان» وجود ندارد**. دسته‌بند فقط از داخل مجموعه برچسب‌هایش جواب می‌دهد؛ ورودی خارج از واژگان بی‌صدا روی هر چه موجود است نگاشته می‌شود. شناختن فضای خروجی مدل به اندازه دقتش مهم است.

</details>

</div>

<div dir="rtl" style="text-align:right">

---
# بخش ۷: ابزارهای حرفه‌ای: آشکارسازی و قطعه‌بندی

دسته‌بندی برای هر تصویر یک سؤال جواب می‌دهد: *این چیست؟* سیستم‌های صنعتی معمولا بیشتر می‌خواهند:

- **آشکارسازی شیء (object detection)**: *چه چیزی کجاست؟* مدل فهرستی از جعبه‌های محصورکننده خروجی می‌دهد، هر کدام با کلاس و امتیاز اطمینان. فناوری پلاک‌خوان‌ها، تشخیص عابر و ربات‌های قفسه‌خوان.
- **قطعه‌بندی معنایی (semantic segmentation)**: *هر پیکسل مال چیست؟* مدل تک‌تک پیکسل‌ها را دسته‌بندی می‌کند و نقاب (mask) می‌سازد. فناوری تحلیل تصویر پزشکی و درک صحنه در خودروی خودران.

هر دو CNN هستند (با سر مخصوص آشکارسازی یا قطعه‌بندی روی ستون‌فقراتی مثل ResNet)، هر دو ازپیش‌آموزش‌دیده و آماده تحویل در `torchvision`. اول **Faster R-CNN**، آموزش‌دیده روی مجموعه COCO (‏۹۱ دسته روزمره، شامل person) (~۱۶۷ مگابایت دانلود):

</div>

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

det_weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
detector = fasterrcnn_resnet50_fpn(weights=det_weights).eval()
det_categories = det_weights.meta["categories"]
print("دسته‌های COCO:", len(det_categories), "| شامل person:", "person" in det_categories)

In [ ]:
import matplotlib.patches as patches

def show_detections(img, threshold=0.7):
    x = torch.tensor(img).permute(2, 0, 1).float()
    with torch.no_grad():
        det = detector([x])[0]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img); ax.axis("off")
    for box, label, score in zip(det["boxes"], det["labels"], det["scores"]):
        if score < threshold:
            continue
        x1, y1, x2, y2 = box
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       fill=False, color="red", linewidth=2))
        ax.text(x1, y1 - 6, f"{det_categories[label]} {score:.2f}",
                color="white", fontsize=9, bbox=dict(facecolor="red", alpha=0.8))
    plt.show()

show_detections(load_image("astronaut.png"))
show_detections(load_image("coffee.png"))

<div dir="rtl" style="text-align:right">

دو نتیجه ارزش خواندن دقیق دارند. تصویر فضانورد، که دسته‌بند ImageNet فقط می‌توانست غلط برچسبش بزند (تمرین ۱۳)، این‌جا درست پردازش می‌شود: **person با اطمینان ۱.۰۰ و جعبه مکانش**، چون مجموعه برچسب COCO این کلاس را دارد و وظیفه آشکارسازی *کجا* را هم پیش‌بینی می‌کند، نه فقط *چه* را. و در عکس قهوه، آشکارساز صحنه را به اجزایش تفکیک می‌کند: میز، فنجان، قاشق‌ها، هر کدام با جعبه و امتیاز خودش. همان ماشین‌آلات کانولوشنی، سر متفاوت، مجموعه برچسب متفاوت؛ انتخاب درست صورت‌بندی مسئله و داده آموزش به اندازه معماری تعیین‌کننده است.

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۱۴
‏`show_detections(load_image("coffee.png"))` را با `threshold=0.5` و با `threshold=0.9` دوباره اجرا کن و هر بار جعبه‌ها را بشمار. آستانه چه چیزی را با چه چیزی معاوضه می‌کند، و قفسه‌خوان فروشگاه چه تنظیمی می‌خواهد در برابر ابزار غربالگری پزشکی؟

</div>

In [ ]:
# ✏️ دو آستانه



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
show_detections(load_image("coffee.png"), threshold=0.5)
show_detections(load_image("coffee.png"), threshold=0.9)
```

آستانه پایین: جعبه‌های بیشتر، شامل تکراری‌ها و حدس‌های کم‌اطمینان (بازیابی بالاتر، صحت پایین‌تر). آستانه بالا: فقط جعبه‌های مطمئن می‌مانند (صحت بالاتر، بازیابی پایین‌تر). درس اسپم جلسه ۹ تعمیم می‌یابد: نقطه کار به هزینه هر نوع خطا بستگی دارد. ابزار غربالگری‌ای که نباید چیزی را از دست بدهد با آستانه پایین کار می‌کند و هشدارهای اشتباه را می‌پذیرد؛ خط لوله تمام‌خودکار بالا می‌رود.

</details>

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۱۵
قطعه‌بندی معنایی با **DeepLabV3** (‏`torchvision.models.segmentation.deeplabv3_resnet50`، ~۱۶۰ مگابایت). با وزن‌های پیش‌فرض بارگذاری‌اش کن، `load_image("astronaut.png")` را از آن عبور بده، `argmax` پیکسل‌به‌پیکسل خروجی را بگیر و نقاب را کنار تصویر نمایش بده. چه کلاس‌هایی در نقاب ظاهر می‌شوند (`seg_weights.meta["categories"]`)؟

</div>

In [ ]:
# ✏️ قطعه‌بندی



<div dir="rtl" style="text-align:right">

<details>
<summary>💡 راه‌حل (برای بازشدن کلیک کن)</summary>

```python
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

seg_weights = DeepLabV3_ResNet50_Weights.DEFAULT
segmenter = deeplabv3_resnet50(weights=seg_weights).eval()
seg_pre = seg_weights.transforms()

img = load_image("astronaut.png")
x = seg_pre(torch.tensor(img).permute(2, 0, 1)).unsqueeze(0)
with torch.no_grad():
    out = segmenter(x)["out"][0]
mask = out.argmax(dim=0)

print("کلاس‌های نقاب:", [seg_weights.meta["categories"][i] for i in mask.unique()])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(img); ax1.set_title("input"); ax1.axis("off")
ax2.imshow(mask); ax2.set_title("per-pixel class"); ax2.axis("off")
plt.show()
```

نقاب، پیکسل‌های `person` را از پس‌زمینه جدا می‌کند: تصمیمی برای هر پیکسل، نه یکی برای هر تصویر یا هر جعبه. دسته‌بندی، آشکارسازی و قطعه‌بندی نردبان استاندارد وظایف بینایی‌اند؛ هر سه همین‌جا، روی همین رایانه، با وزن‌های ازپیش‌آموخته اجرا شدند.

</details>

</div>

<div dir="rtl" style="text-align:right">

---
# بخش ۸: جمع‌بندی و گام‌های بعدی

نتایج جلسه در یک جدول:

| وظیفه | مدل | داده مصرفی | نتیجه |
|---|---|---|---|
| دسته‌بندی | تمام‌متصل (جلسه ۷) | ۵۰هزار تصویر | ۰.۴۹۹ |
| دسته‌بندی | CNN از صفر | ۵۰هزار تصویر | ≈ ۰.۶۹ |
| دسته‌بندی | ویژگی ResNet18 + خطی | ۵هزار تصویر | ≈ ۰.۸۲ |
| دسته‌بندی | ResNet18 ریزتنظیم‌شده | ۵هزار تصویر | ≈ ۰.۸۷ |
| آشکارسازی | Faster R-CNN (ازپیش‌آموخته) | صفر از ما | person ۱.۰۰، با جعبه |
| قطعه‌بندی | DeepLabV3 (ازپیش‌آموخته) | صفر از ما | نقاب پیکسل‌به‌پیکسل |

استدلال از ابتدا تا انتها: کانولوشن یک جمع وزن‌دار محلی با وزن مشترک است (در numpy پیاده شد)؛ لایه‌های کرنل یادگرفتنی سلسله‌مراتب ویژگی می‌سازند (آموزش داده و بازرسی شد)؛ این معماری با یک‌سوم پارامتر، شبکه تمام‌متصل را شکست می‌دهد؛ داده‌افزایی داده را با تنوع حافظ برچسب گسترش می‌دهد؛ ستون‌فقرات‌های ازپیش‌آموخته ویژگی را به کالای آماده تبدیل می‌کنند، چنان‌که دسته‌بند خطی با یک‌دهم داده برنده می‌شود و یک ریزتنظیم کوتاه باز هم جلوتر می‌رود؛ و با سر مناسب، همان ستون‌فقرات‌ها آشکارسازی و قطعه‌بندی را در کیفیت صنعتی حل می‌کنند.

</div>

<div dir="rtl" style="text-align:right">

### ✏️ تمرین ۱۶: تکلیف
۱. ‏CNN سه‌بلوکه تمرین ۷ را از دقت آزمون **۰.۷۵** عبور بده (آموزش طولانی‌تر، کرنل بیشتر، هر دو). هر تغییر و اثر اندازه‌گیری‌شده‌اش را ثبت کن.

۲. مسابقه [Digit Recognizer](https://www.kaggle.com/competitions/digit-recognizer) در Kaggle: ارسال جلسه ۷ (`MLPClassifier`) را با یک CNN کوچک جایگزین کن و نمره‌های جدول را مقایسه کن.

۳. روی GPU در Colab، ریزتنظیم بخش ۶ را این بار با خط لوله داده‌افزایی بخش ۵ داخل loader آموزش و ۳ ایپاک یا بیشتر اجرا کن. اندازه بگیر که می‌توانی از ۰.۹۰ عبور کنی یا نه.

*جلسه بعد: همین سفر برای متن، از واژه‌های عددشده تا مدل‌هایی که می‌خوانند.*

</div>